In [1]:
import json
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings # pip install sentence-transformers
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
# from langchain_groq import ChatGroq
from openai import AzureOpenAI
from FlagEmbedding import FlagReranker
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

from concurrent.futures import ThreadPoolExecutor

In [2]:
api_key = 'xxx'
endpoint = 'ooo'
deployment = 'gpt-4o'
api_version='2024-12-01-preview'

client = AzureOpenAI(
    api_key = api_key,
    azure_endpoint = endpoint,
    api_version = api_version
)

In [3]:
def normalize_medical_note(raw_text: str) -> str:
    """
    Normalize (+)/(-) polarity without medical interpretation.
    """
    normalized_lines = []

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # Replace (+) and (-) with explicit polarity
        # line = re.sub(r"\(\+\)", ": Yes", line)
        # line = re.sub(r"\(-\)", ": No", line)

        line = re.sub(r"([a-zA-Z0-9_]+)\s*\(\+\)", r"Yes \1", line)
        line = re.sub(r"([a-zA-Z0-9_]+)\s*\(-\)", r"No \1", line)

        normalized_lines.append(line)

    return "\n".join(normalized_lines)


In [4]:
def summarize(raw_text: str):
    prompt = f"""
    
Task:
From the following outpatient clinical note, extract ONLY information relevant to medical imaging decision-making.

Focus on clinical features that may influence whether imaging is indicated and which type of imaging may be appropriate.


General Rules:

Ignore laboratory values unless they directly indicate infection, inflammation, bleeding risk, or other conditions that could influence imaging decisions.

Ignore purely administrative, social, psychiatric, or medication details unless they suggest:
• acute neurologic change
• structural disease
• trauma
• infection
• malignancy
• postoperative complications

Include only information that may affect imaging decisions, such as:

• Key symptoms (e.g., pain, neurologic deficit, trauma, mass, fever, focal deficits)
• Symptom onset and progression
• Relevant medical history (e.g., malignancy, trauma, surgery, infection)
• Physical examination findings related to structural pathology
• Red-flag symptoms
• Prior imaging findings
• Any explicit or implicit indication that imaging may be required


Red Flag Definition:

Red flag symptoms are specific symptoms or physical signs indicating the possible presence of serious or potentially life-threatening disease that may require urgent imaging evaluation.


Diagnostic Language Interpretation Rules:

When interpreting diagnostic phrases in radiology or clinical notes, strictly distinguish the following categories:

1. Confirmed disease  
Terms: "confirmed", "proven", "diagnosed"  
→ Record as confirmed pathology.

2. Suspected disease  
Terms: "suspicious for", "highly suggestive of", "likely"  
→ Record as suspected pathology.

3. Possible disease  
Terms: "possible", "cannot exclude", "may represent"  
→ Record as possible pathology.

4. Rule-out consideration  
Terms: "rule out", "should be ruled out", "to exclude"  
→ Do NOT convert to suspected disease.  
→ Record only as diagnostic consideration.


Important Constraints:

1. Do NOT add interpretations, diagnoses, or facts that are not present in the clinical note.
2. All extracted information must strictly reflect the source text.
3. Use concise bullet points.


Clinical Information Extraction Procedure:


Step 1 — Clinical Entity Extraction

Identify and list ALL clinically relevant medical entities mentioned in the note that may influence imaging decisions.

Include entities such as:

• symptoms  
• diseases or diagnoses  
• structural abnormalities  
• fractures or injuries  
• tumors or masses  
• anatomic abnormalities  
• neurologic deficits  
• prior imaging findings  
• prior surgeries or treatments  

List entities exactly as stated in the note.


Step 2 — Entity Categorization

Classify the extracted entities into the following categories:

Symptoms:
(e.g., pain, weakness, numbness, headache)

Structural Abnormalities:
(e.g., fracture, vertebral compression fracture, deformity, disc herniation, spondylolisthesis, spinal canal stenosis, mass)

Prior Imaging Findings:
(abnormal findings reported on prior imaging)

Diagnosed or Suspected Pathology:
(e.g., tumor, infection, hemorrhage)

Neurologic Findings:
(e.g., radiculopathy, focal neurologic deficit)


Structural Preservation Rule:

If a structural abnormality is identified in prior imaging or examination
(e.g., fracture, deformity, mass, disc herniation, stenosis),
it MUST be preserved in the structured summary even if it is not the primary symptom.

Structural abnormalities often influence imaging modality selection and therefore must not be omitted.


Importance Rules:

1. Track the chronological order of events if dates are present.
2. Track each symptom longitudinally across dates.
3. Expand medical abbreviations into full medical terminology when possible.
4. Internally track symptom resolution or persistence using the latest dated entry.
5. Only output the final structured result (do not output reasoning steps).


Special Case Handling:

Step 1:
If prior imaging has already revealed a structural lesion, tumor suspicion, fracture, or other significant abnormality, prioritize lesion-related clinical scenarios rather than symptom-based scenarios.

Step 2:
If surgery, radiation therapy, or other major treatment has been performed:

• If the treatment occurred within the past 6 months → treat as follow-up imaging context  
• If more than 6 months → evaluate based primarily on current symptoms


Clinical note:
{raw_text}


Output Format:

Sex:
Age:


History (chronological):
- YYYY/MM/DD: symptom or clinical event
- ...


Pathology:
confirmed_pathology:
suspected_pathology:
differential_diagnosis:


Structural Abnormalities:
(list structural lesions such as fractures, deformities, masses, or other anatomical abnormalities)


Prior Imaging Findings:
(abnormal findings reported on prior imaging studies)


Current Symptoms Status:

remaining:
improving/resolving:
resolved:


Potential Imaging Indicators
(based on remaining symptoms AND structural findings):
-
-


Summary:

age_group: child / adult / elderly

primary_clinical_presentation:
(e.g., headache, abdominal pain, trauma, focal neurologic deficit, joint pain)

suspected_pathology_category:
(neurologic / musculoskeletal / abdominal / thoracic / oncologic / infectious / traumatic / unclear)

anatomical_region:
(head / brain / spine / lumbar spine / abdomen / thorax / extremity)

inflammatory_or_infectious_process:
True / False / Unclear

clinical_phase:
initial evaluation / follow-up / surveillance / unclear

red_flag_present:
True / False

imaging_decision_context:
(symptom-based / post-treatment / suspected malignancy / trauma / infection)

"""

    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(model=deployment, messages=messages, temperature=0.1, top_p=0.9, max_tokens=3000)
    return response.choices[0].message.content.strip()

In [5]:
def structured_to_acr_indication(structured_case: str):
    prompt = f"""

Task
Convert the structured clinical case into a **single-line ACR-style imaging indication** suitable for radiology guideline retrieval.


Output Requirements

• Produce **one single sentence only**.
• Use **clear radiology clinical language similar to ACR Appropriateness Criteria indications**.
• The sentence must begin with **age and sex**.
• Do not output explanations.


Sentence Construction Order

Construct the sentence using the following priority order whenever the information is available:

1. Age and sex
2. Primary clinical presentation (main symptom or chief complaint)
3. Key associated symptoms or neurologic findings
4. Structural abnormalities (if present)
5. Known or suspected pathology
6. Clinical course or modifiers
7. Imaging purpose or stage


Core Sentence Template

Prefer the following template:

[age]-year-old [sex] with [primary symptom ± associated symptoms], [structural abnormality if present], [known or suspected pathology], [clinical context], [imaging purpose or stage].

Example:

"70-year-old female with persistent low back pain radiating to the left leg and L5 radiculopathy, known vertebral compression fracture at T12, suspected metastatic spinal lesion at L4, follow-up imaging evaluation."


Structural Findings Rule (Important)

If the structured case contains **any structural abnormality**, it MUST be preserved in the sentence.

Structural abnormalities include:

• fracture  
• vertebral compression fracture  
• mass or tumor  
• deformity  
• disc herniation  
• spondylolisthesis  
• spinal canal stenosis  
• abscess  
• hematoma  
• other anatomical abnormalities  

These findings should appear **after symptoms but before imaging purpose**.

If multiple structural abnormalities exist, include the most clinically relevant one or two.


Preserve Important Clinical Information

Retain the following information whenever available:

• Age and sex  
• Primary symptom or chief complaint  
• Important associated symptoms  
• Structural abnormalities  
• Known or suspected pathology  
• Disease progression (worsening, progressive symptoms)  
• Imaging stage (initial evaluation, follow-up imaging, post-treatment evaluation)


Red Flag Integration

If red-flag features are present, incorporate them naturally using clinical language.

Examples include:

• neurologic deficit  
• trauma  
• cancer history  
• immunocompromise  
• fever  
• rapidly progressive symptoms  

Do NOT explicitly write the phrase "red flag".


Clinical Course

Preserve information describing the clinical course when present:

• acute  
• chronic  
• persistent  
• progressive  
• failed conservative therapy  
• recurrence  
• post-treatment status


Symptoms Rule

• Include **active symptoms only**.
• If a symptom was previously present but resolved, append "(resolved)".


Medical Terminology

• Expand abbreviations into full clinical terminology when possible.
• Prefer formal terminology commonly used in radiology indications.


Do NOT Include

Do not include:

• dates  
• administrative metadata  
• feature-engineering variables (e.g., age_group, inflammatory=True)


No Hallucination Rule

Do not introduce new diagnoses or clinical findings that are not explicitly present in the structured case.


Structured case:
{structured_case}


Output:
(one single sentence only)
"""

    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        temperature=0.1,
        top_p=0.9,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

In [52]:
def reason_filter(patient_summary: str, variant_text: str):
    prompt = f"""

你現在是一位需要評估患者病歷與 ACR Variant 適合度的專業專科醫師，
僅討論病人本身的情況，判斷每組病例與 variant 之間是否「適合 / 不適合」，並提供主要理由。

Patient case:
{patient_summary}

Variant description:
{variant_text}


Task:

Step 1. Extract the key clinical findings from the patient case.

Include:
- primary symptom
- important associated findings
- known etiology or pathology
- red flags
- imaging phase (initial vs follow-up)

If a category is not mentioned in the case, write "None".

注意：
若病例中的問題已標記為 resolved，則不納入判斷。


Step 2. Identify the key clinical condition or diagnostic scenario targeted by the variant.

Summarize the diagnostic focus of the variant in **one short sentence**.


Step 3. Compare the patient case with the variant scenario.

List:

A. Matching findings  
Findings from the patient that directly support the variant indication.

B. Missing required findings  
Conditions explicitly required by the variant but not present in the case.

C. Conflicting findings  
A contradiction indicates that the variant's information contradicts the information in the medical record. 
If the variant does not mention the relevant context, it should be considered 'Missing required findings'.

Rules:
- Only use information explicitly stated in the patient case.
- Do NOT assume or infer missing symptoms.
- red_flag_present 並不等於 cancer、infection 或 immunosuppression，只有病例明確描述這些疾病或危險因子時才算符合。
- The red flag in variants doesn't need to match all of them; it's enough if the case mentions just one.
- Take care about age group in patient case and variant.


Step 4. Clinical judgment

Determine whether this variant is appropriate for the patient.

Decision rules:

1. Variant中若包含特定疾病或條件（例如 cancer、infection、immunosuppression、trauma、stroke、drug exposure 等），
   病例中必須明確提及該疾病或合理懷疑，否則判定為不適合。

2. 若variant包含特定評分或指標（例如 GCS、clinical decision rule 等），
   病例沒有提供相關資訊時，直接判定為不適合。

3. 若該variant有"suspicion of cancer, infection, or immunosuppression"，
   病例必須明確提到其中之一，否則即使其他症狀符合，也判定為不適合。

4. 承3.，反之，若病例有提及某些症狀與感染等醫學描述，但該variate沒有提到，仍可以判為適合。

5. 若該variant有"One or more of the following:"，則該病例必須滿足至少一項的描述，才可以被判為適合。

6. 承5.，若該variant已滿足該病例至少一項描述，則可以判為適合。

7. 承5.，若該variant有"without any of the following"，則該病例不能有任何符合的描述，否則判為不適合。

8. 每個variant的子句中，若以or連接各描述，僅需符合其中一個描述，該子句就被滿足；若以and連接各描述，需全部符合所有描述，該子句才被滿足。

9. red_flag_present 並不等於 cancer、infection 或 immunosuppression。只有病例明確描述這些疾病或相關危險因子時才算符合。

10. 若病例中的問題已標記為 resolved，則不納入判斷。

11. 若variant內容與病例的主要疾病、症狀或病程一致，但僅有 "initial imaging / follow-up imaging" 不完全一致，仍判為適合。

12. 若 patient_summary 提到 red-flag，但 variant 指出 No red flag（或反之），則判為不適合。

13. 若該病例有提供不同部位的詳細描述(例如:一個人同時有胸椎和腰椎的問題)，則該variant不用同時包含兩者的描述，只要其中一個滿足以上規則即可。

14. 「主要理由」優先討論順序：
     疾病 → 病因 → 症狀 → red flag → imaging phase。

15. Key Word:主要理由中，該variant與patient case相同的關鍵字，但"關鍵字"僅限症狀或疾病等醫學相關名詞(以patient case的原文輸出)。

16. 承12.13.，主要理由和Key Word盡量不要只有Headache、Low back pain、red flag等topic等級的描述，可以多一點病例與該variant都相符的細節。


Output must be concise and written in Chinese.


Output format:


發現:
A. Matching findings:
B. Missing required findings: 
C. Conflicting findings:

病例關鍵資訊:
- primary symptom:
- associated findings:
- etiology/pathology:
- red flags:
- imaging phase:

**綜合評估: 適合 / 不適合**

主要理由:
(一句話，優先順序：疾病 → 病因 → 症狀 → red flag → imaging phase)

Key Word:
"""
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": "請根據上面的病例給出 variant 建議。"}
    ]

    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        temperature=0.1,
        top_p=0.9,
        max_tokens=3000
    )

    return response.choices[0].message.content.strip()

In [53]:
def process_reason_filter(args):
    """封裝單次的 API 呼叫與錯誤處理"""
    i, doc_score, query_acr_style = args
    doc, score = doc_score
    variant_text = doc.page_content
    
    try:
        # 執行原本的 reason_filter
        reason_output = reason_filter(
            patient_summary=query_acr_style,
            variant_text=variant_text
        )
        return {
            "index": i,
            "content": variant_text,
            "reason": reason_output,
            "success": True
        }
    except Exception as e:
        return {
            "index": i,
            "content": variant_text,
            "reason": f"[錯誤] reason_filter 失敗：{e}",
            "success": False
        }


In [54]:
# 載入 ACR variants

with open(r'C:\Users\482525\Work_File\放射科RAG_model\acr_all_variants.json', 'r', encoding='utf-8') as f:
    acr_data = json.load(f)

# documents = []
# for item in acr_data:
#     if isinstance(item, dict) and 'variant_description' in item:
#         doc = Document(
#             page_content=item['variant_description'],
#             metadata={
#                 "procedures": item.get('procedures', []),
#                 "topic": item.get('topic', '')
#             }
#         )
#         documents.append(doc)

# 檢索 + procedure, topic

documents = []

for item in acr_data:
    if isinstance(item, dict) and 'variant_description' in item:

        procedures_text = "\n".join(
            [f"{p['category']}: {p['procedure']}" 
             for p in item.get('procedures', [])
             if p["category"] in ["Usually Appropriate", "May Be Appropriate"]]
        )

        full_text = f"""
Topic: {item.get('topic', '')}

Variant:
{item['variant_description']}

Recommended Procedures:
{procedures_text}
"""

        doc = Document(
            page_content=full_text,
            metadata={
                "topic": item.get('topic', '')
            }
        )

        documents.append(doc)

print(f"載入 {len(documents)} 個 variants")

載入 1278 個 variants


In [55]:
# print(documents[0].page_content)

In [10]:
# Embedding + FAISS
# 選項 A: bge-m3、B: nomic-embed-text-v1.5

local_model_path = r"C:\Users\482525\Desktop\bge-m3-local" # bge-m3-local / sentence-bert-base_local / text-embedding-3-large

embeddings = HuggingFaceEmbeddings(
    model_name=local_model_path,               
    model_kwargs={'device': 'cpu'},  # cuda
    encode_kwargs={'normalize_embeddings': True} 
)

# 儲存 index，下次直接載入
# vectorstore = FAISS.from_documents(documents, embeddings)
# vectorstore.save_local(r"C:\Users\482525\Work_File\embedding_model\faiss_acr_index_adapt")
vectorstore = FAISS.load_local(r"C:\Users\482525\Work_File\embedding_model\faiss_acr_index_adapt", embeddings, allow_dangerous_deserialization=True)

C:\Users\482525\AppData\Local\Temp\ipykernel_14304\1820803499.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [11]:
# query_file_path = Path(r'C:\Users\482525\Desktop\SOA_病例_新竹\test_headache\病歷號碼 0702455184.txt'.strip('\u202a')) # utf-8
query_file_path = Path(r'‪C:\Users\482525\Desktop\SOA_病例_新竹\test_low_back_pain\病歷號碼 0014563146.txt'.strip('\u202a'))

with open(query_file_path, 'r', encoding='utf-8') as f:
    query = f.read().strip()

print(f"--- Query 內容 ---")
# print(query)
print("-" * 30)
    
raw = query
raw = normalize_medical_note(raw)
structured_case = summarize(raw)
print(structured_case)

--- Query 內容 ---
------------------------------
### Extracted Information

---

**Sex:**  
Male  

**Age:**  
82  

---

### History (chronological):  
- **2013/08/26:** Post right L3-4 posterior fusion surgery.  
- **2025/04/25:** Low back pain off and on for 6 months; walking disturbance with left leg weakness.  
- **2025/05/13:** Right peroneal nerve conduction velocity test (43.4 m/s).  
- **2025/05/23:** Patient refused admission despite aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

---

### Pathology:  

**Confirmed Pathology:**  
- Status post L3-4 posterior fusion surgery.  
- Marked scoliosis of thoracic and lumbar spine.  
- Retrolisthesis at L2-3 and L5-S1.  
- Annular bulging discs at L1-2, L2-3, L4-5, and L5-S1.  
- Bilateral neuroforaminal encroachment at L1-2 and L2-3.  
- Bilateral neuroforaminal narrowing at L4-5.  
- Right neuroforaminal narrowing at L5-S1.  
- Mixed Modic type I and II endplate degeneration of L1, L2, L4, and L5 vertebral

In [12]:
query_acr_style = structured_to_acr_indication(structured_case)
print(query_acr_style)

"82-year-old male with persistent low back pain, walking disturbance, and left leg weakness, marked scoliosis of thoracic and lumbar spine, retrolisthesis at L2-3 and L5-S1, annular bulging discs, bilateral neuroforaminal encroachment and narrowing, Modic endplate changes, and aneurysmal dilatation of abdominal aorta with mural thrombus, follow-up imaging evaluation."


In [56]:
structured_case = '''
**Sex:**  
Male  

**Age:**  
82  

---

### History (chronological):  
- **2013/08/26:** Post right L3-4 posterior fusion surgery.  
- **2025/04/25:** Low back pain off and on for 6 months; walking disturbance with left leg weakness.  
- **2025/05/13:** Right peroneal nerve conduction velocity test (43.4 m/s).  
- **2025/05/23:** Patient refused admission despite aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

---

### Pathology:  

**Confirmed Pathology:**  
- Status post L3-4 posterior fusion surgery.  
- Marked scoliosis of thoracic and lumbar spine.  
- Retrolisthesis at L2-3 and L5-S1.  
- Annular bulging discs at L1-2, L2-3, L4-5, and L5-S1.  
- Bilateral neuroforaminal encroachment at L1-2 and L2-3.  
- Bilateral neuroforaminal narrowing at L4-5.  
- Right neuroforaminal narrowing at L5-S1.  
- Mixed Modic type I and II endplate degeneration of L1, L2, L4, and L5 vertebral bodies.  
- Aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

**Suspected Pathology:**  
None explicitly stated.  

**Differential Diagnosis:**  
None explicitly stated.  

---

### Structural Abnormalities:  
- Marked scoliosis of thoracic and lumbar spine.  
- Retrolisthesis at L2-3 and L5-S1.  
- Annular bulging discs at L1-2, L2-3, L4-5, and L5-S1.  
- Bilateral neuroforaminal encroachment at L1-2 and L2-3.  
- Bilateral neuroforaminal narrowing at L4-5.  
- Right neuroforaminal narrowing at L5-S1.  
- Mixed Modic type I and II endplate degeneration of L1, L2, L4, and L5 vertebral bodies.  
- Status post L3-4 posterior fusion surgery.  
- Aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

---

### Prior Imaging Findings:  
- **2025/05/13:** Right peroneal nerve conduction velocity test (43.4 m/s).  
- **2025/05/23:** Thoracolumbar spine MRI:  
  - Annular bulging discs at L1-2, L2-3, L4-5, and L5-S1.  
  - Bilateral neuroforaminal encroachment at L1-2 and L2-3.  
  - Bilateral neuroforaminal narrowing at L4-5.  
  - Right neuroforaminal narrowing at L5-S1.  
  - Mixed Modic type I and II endplate degeneration of L1, L2, L4, and L5 vertebral bodies.  
  - Status post L3-4 posterior fusion surgery.  
  - Aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

---

### Current Symptoms Status:  

**Remaining:**  
- Low back pain off and on for 6 months.  
- Walking disturbance with left leg weakness.  

**Improving/Resolving:**  
None explicitly stated.  

**Resolved:**  
None explicitly stated.  

---

### Potential Imaging Indicators:  
- Persistent low back pain for 6 months.  
- Walking disturbance with left leg weakness.  
- Structural abnormalities including scoliosis, retrolisthesis, annular bulging discs, neuroforaminal encroachment/narrowing, and Modic changes.  
- Aneurysmal dilatation of abdominal aorta with mural thrombus formation.  

---

### Summary:  

**Age Group:**  
Elderly  

**Primary Clinical Presentation:**  
Low back pain, walking disturbance, left leg weakness.  

**Suspected Pathology Category:**  
Musculoskeletal and vascular.  

**Anatomical Region:**  
Lumbar spine, thoracic spine, abdomen.  

**Inflammatory or Infectious Process:**  
Unclear.  

**Clinical Phase:**  
Follow-up.  

**Red Flag Present:**  
True (persistent pain, neurologic deficit, aneurysmal dilatation of abdominal aorta).  

**Imaging Decision Context:**  
Symptom-based and post-treatment.

'''
query_acr_style = '''
82-year-old male with persistent low back pain, walking disturbance, and left leg weakness, marked scoliosis of thoracic and lumbar spine, retrolisthesis at L2-3 and L5-S1, annular bulging discs, bilateral neuroforaminal encroachment and narrowing, Modic endplate changes, and aneurysmal dilatation of abdominal aorta with mural thrombus, follow-up imaging evaluation.
'''

In [57]:
query_instruction_for_rerank = """
Rank ACR imaging guideline variants by how well they match the clinical scenario.
Prioritize matching of symptoms, anatomical location, and structural abnormalities.
"""

k = 20
fetch_k = 100

candidates = vectorstore.similarity_search(query_acr_style, k=fetch_k)
# print(candidates[0].page_content)

In [58]:
reranker = FlagReranker(
    r"C:\Users\482525\Desktop\bge-reranker-v2-m3_local",
    use_fp16=True,
    batch_size=32,
    query_instruction_for_rerank=query_instruction_for_rerank,
    query_instruction_format="Instruction: {}\nClinical Case: {}",
    passage_instruction_format="ACR Variant: {}",
    devices=['cpu', 'cpu']
)

pairs = [[query_acr_style, doc.page_content] for doc in candidates]
scores = reranker.compute_score(pairs, normalize=True)
scored_docs = list(zip(candidates, scores))

# 重新排序
scored_docs.sort(key=lambda x: x[1], reverse=True)
top_docs = scored_docs[:k]

print(f"\nTop {k} reranked variants：")

for i, (doc, score) in enumerate(top_docs, 1):
    print(f"{i}. score={score:.4f}")
    print(doc.page_content)
    print("-" * 50)

Chunks: 100%|███████████████████████████████████████████████████████████████████████████| 2/2 [04:05<00:00, 122.92s/it]


Top 20 reranked variants：
1. score=0.3104

Topic: Low Back Pain

Variant:
Subacute or chronic low back pain with or without radiculopathy. No red flags. No prior management. Initial imaging.

Recommended Procedures:


--------------------------------------------------
2. score=0.3061

Topic: Low Back Pain

Variant:
Subacute or chronic low back pain with or without radiculopathy. Surgery or intervention candidate with persistent or progressive symptoms during or following 6 weeks of optimal medical management. Initial imaging.

Recommended Procedures:
Usually Appropriate: MRI lumbar spine without IV contrast
May Be Appropriate: Radiography lumbar spine
May Be Appropriate: MRI lumbar spine without and with IV contrast
May Be Appropriate: Bone scan whole body with SPECT or SPECT/CT complete spine
May Be Appropriate: CT lumbar spine without IV contrast
May Be Appropriate: CT myelography lumbar spine

--------------------------------------------------
3. score=0.2853

Topic: Low Back Pain


In [59]:
patient_summary=structured_case
variant_text = doc.page_content

# 設定平行數量 (k)，建議 5-10
k = 10 
tasks = [(i, doc, structured_case) for i, doc in enumerate(top_docs, 1)]

with ThreadPoolExecutor(max_workers=k) as executor:
    results = list(executor.map(process_reason_filter, tasks))
    
for res in results:
    print(f"Variate {res['index']}: {res['content']}")
    print("\n--- Filter Reason ---")
    print(res['reason'])
    print("=" * 80)

Variate 1: 
Topic: Low Back Pain

Variant:
Subacute or chronic low back pain with or without radiculopathy. No red flags. No prior management. Initial imaging.

Recommended Procedures:



--- Filter Reason ---
發現:  
A. Matching findings:  
- Subacute or chronic low back pain.  
- Radiculopathy (walking disturbance with left leg weakness).  

B. Missing required findings:  
- No red flags (variant要求無red flags，但病例中有red flags)。  
- No prior management (病例中有既往手術史和多次影像檢查，屬於有prior management)。  

C. Conflicting findings:  
- Red flags present in the case (aneurysmal dilatation of abdominal aorta, persistent pain, neurologic deficit), which contradicts the variant's "No red flags" requirement.  

病例關鍵資訊:  
- primary symptom: Low back pain.  
- associated findings: Walking disturbance with left leg weakness, structural abnormalities (e.g., scoliosis, retrolisthesis, neuroforaminal narrowing), aneurysmal dilatation of abdominal aorta.  
- etiology/pathology: Degenerative spine changes, prior L3

In [60]:
variate_blocks = []

for res in results:

    variate_key = f"variate {res['index']}"

    decision_match = re.search(r"綜合評估.*?(適合|不適合)", res['reason'])
    final_decision = decision_match.group(1) if decision_match else "未知"

    reason_match = re.search(r"主要理由[:：]\s*(.+)", res['reason'])
    main_reason = reason_match.group(1).strip() if reason_match else ""

    key_word_match = re.search(r"Key Word[:：]\s*(.+)", res['reason'])
    key_word = key_word_match.group(1).strip() if key_word_match else ""

    block = {
        "variate": res['content'],
        "decision": final_decision,
        "main_reason": main_reason,
        "key_word": key_word,
        "text": res['reason']
    }

    variate_blocks.append(block)

In [61]:
def parse_variate_result(block):

    if block.get("decision") != "適合":
        return None

    return {
        "variate": block["variate"],
        "decision": block["decision"],
        "main_reason": block["main_reason"],
        "key_word": block["key_word"],
    }

In [62]:
results = []

for b in variate_blocks:
    r = parse_variate_result(b)
    if r:
        results.append(r)

for i, r in enumerate(results, 1):

    print(f"{i}:")
    print(r["variate"])
    print()
    print(f"綜合評估: {r['decision']}")
    print(f"主要理由: {r['main_reason']}")
    print(f"Key Word: {r['key_word']}")
    print("\n" + "-"*60 + "\n")

1:

Topic: Low Back Pain

Variant:
Subacute or chronic low back pain with or without radiculopathy. Surgery or intervention candidate with persistent or progressive symptoms during or following 6 weeks of optimal medical management. Initial imaging.

Recommended Procedures:
Usually Appropriate: MRI lumbar spine without IV contrast
May Be Appropriate: Radiography lumbar spine
May Be Appropriate: MRI lumbar spine without and with IV contrast
May Be Appropriate: Bone scan whole body with SPECT or SPECT/CT complete spine
May Be Appropriate: CT lumbar spine without IV contrast
May Be Appropriate: CT myelography lumbar spine


綜合評估: 適合
主要理由: 該病例符合「Subacute or chronic low back pain with radiculopathy」且有持續症狀與手術史，適合進行初步影像檢查。
Key Word: Low back pain, radiculopathy, persistent symptoms, surgery candidate.

------------------------------------------------------------

2:

Topic: Low Back Pain

Variant:
Low back pain with history of prior lumbar surgery and with or without radiculopathy. New or pro